# G0-exact - reproduce the original hub protocol, line for line

**Run this instead of the earlier G0.** Every construction choice is copied from `G4_cross_lineage.ipynb`: four spaces, per-space scaling by mean column std, random seed-0 split, bge targets, ridge alpha 1e-2.

**The gate is SigLIP returning 94.2%** within 1.0 point. If it does, the transfer numbers become LEVELS comparable to C.13.2's band and the protocol-offset caveat can be withdrawn. If not, the calibrated comparison stands as the honest fallback.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G0-exact — reproduce the ORIGINAL hub protocol, line for line.
# Supersedes the earlier reconstruction. Run this instead of G0.
#
# WHY THIS REPLACES THE RECONSTRUCTION. The first rebuild approximated the
# hub because the original construction was not to hand: it used seven raw
# spaces, a sequential split, SBERT targets and ridge alpha 1.0. It passed
# a spectrum-shape gate at 0.71 per cent but the retrieval protocol came
# out 7.8 points low, which forced every transfer number to be quoted as
# an ordering rather than a level.
#
# G4_cross_lineage.ipynb contains the actual construction, so there is no
# longer any reason to approximate. Every choice below is copied from it:
#
#   spaces      img_small, img_base, img_large (DINOv2 cls+patch) + txt_bge
#               - FOUR, not seven
#   rows        N_PAIRS = min row count across the DINOv2 caches, then the
#               first N_PAIRS rows of each (the deterministic prefix E1 used)
#   scaling     per space: (X - mean) / (mean of per-column std)
#   split       rng(0).permutation, first 1000 eval, rest train - RANDOM,
#               not sequential
#   basis       SVD of the centred concat; V[:512].T / (sigma / sqrt(n))
#   entry maps  ridge, alpha 1e-2, from each space into HUB_TR
#   head        ridge, alpha 1e-2, from img_small's hub coords to txt_bge
#   gallery     L2-normalised txt_bge on the eval rows
#
# THE GATE, and it is a real one. SigLIP 2 has a PUBLISHED value of 94.2
# per cent of native under this exact protocol. If this notebook returns
# 94.2, the protocol is reproduced and ConvNeXt's number is directly
# comparable to the 93.8-96.5 per cent band - no offset, no caveat, no
# "ordering only". If it does not, the reproduction has failed and the
# earlier calibrated comparison stands as the honest fallback.
#
# Pre-registered tolerance: within 1.0 point of 94.2. Anything looser
# would let a near-miss pass as a reproduction.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])

HUB_DIM, ALPHA, TRAIN_ON = 512, 1e-2, "img_small"
N_EVAL, SEED = 1000, 0
PUB_SIGLIP = 0.942          # G4's published cross-lineage figure
PUB_BASE, PUB_LARGE = 0.959, 0.929   # C.13 within-family numbers
BAND = (0.938, 0.965)
TOL = 0.010                 # 1.0 point

HELD_OUT = {
    "SigLIP 2": ("e1_img_ckpt_siglip2-base-patch16-224_native.npz", PUB_SIGLIP),
    "ConvNeXt": ("e1_img_ckpt_convnext-base-224-22k_native.npz", None),
}

In [ ]:
# ---------- 1. the four spaces, exactly as G4 loaded them ----------
SPACES = {}
for size in ("small", "base", "large"):
    f = DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz"
    assert f.exists(), f"missing {f.name}"
    SPACES[f"img_{size}"] = np.load(str(f))["img"].astype(np.float64)

N_PAIRS = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N_PAIRS] for k, v in SPACES.items()}

d = np.load(str(DATA_DIR / "crossmodal_pairs.npz"))
SPACES["txt_bge"] = d["txt"].astype(np.float64)[:N_PAIRS]

print(f"N_PAIRS = {N_PAIRS} (narrowest DINOv2 cache)")
for k, v in SPACES.items():
    print(f"  {k:10s} {v.shape}  mean row norm {np.linalg.norm(v, axis=1).mean():7.2f}")
print("\nNOTE: rows are aligned by POSITIONAL PREFIX, which is what the")
print("original did - every cache was written over the same deterministic")
print("id list. No keep-array join is involved, so this reproduces the")
print("original's alignment assumption rather than substituting a new one.")

rng = np.random.default_rng(SEED)
perm = rng.permutation(N_PAIRS)
te, tr = perm[:N_EVAL], perm[N_EVAL:]
print(f"split: {len(tr)} train / {len(te)} eval, random with seed {SEED}")


def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)


def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)


def recall(S):
    o = np.argsort(-S, 1)
    r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

In [ ]:
# ---------- 2. the hub, exactly as G4 built it ----------
# per-space scaling by the MEAN of the per-column standard deviations -
# one scalar per space, not per-column whitening. This is the step the
# earlier reconstruction omitted, and it is why row norms spanning 234x
# did not wreck the original concat.
_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_u, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
BASIS = _VT[:HUB_DIM].T / (_sv[:HUB_DIM] / np.sqrt(len(_ref)))
HUB_TR = (_ref - _mu) @ BASIS

TO_HUB = {k: ridge(SPACES[k][tr], HUB_TR) for k in SPACES}
HEAD = ridge(SPACES[TRAIN_ON][tr] @ TO_HUB[TRAIN_ON], SPACES["txt_bge"][tr])
GAL = l2n(SPACES["txt_bge"][te])

print(f"\nhub: {HUB_DIM}-d from a {_ref.shape[1]}-d concat of "
      f"{len(SPACES)} spaces; head trained on {TRAIN_ON}")
print(f"  singular values at widths 64/128/256/512: "
      + " ".join(f"{_sv[w-1]:8.2f}" for w in (64, 128, 256, 512)))

In [ ]:
# ---------- 3. the within-family numbers, as a first check ----------
print("\n" + "=" * 70)
print(f"{'encoder':22s}{'R@1':>9}{'native':>9}{'% native':>11}{'published':>11}")
print("=" * 70)
native = {}
for enc in [k for k in SPACES if k.startswith("img_")]:
    P = l2n((SPACES[enc][te] @ TO_HUB[enc]) @ HEAD)
    nat = ridge(SPACES[enc][tr], SPACES["txt_bge"][tr])
    rn = recall(l2n(SPACES[enc][te] @ nat) @ GAL.T)
    r = recall(P @ GAL.T)
    pct = r[1] / max(rn[1], 1e-9)
    native[enc] = pct
    pub = {"img_base": PUB_BASE, "img_large": PUB_LARGE}.get(enc)
    if enc == TRAIN_ON and pct > 1.0:
        print(f"  NOTE: {enc} exceeds its own native fit ({pct:.1%}). The hub"
              f" is a 512-d bottleneck between a {SPACES[enc].shape[1]}-d"
              f" source and the target, so it regularises where the direct"
              f" ridge does not. Expected, not an error - but it means"
              f" 'per cent of native' can exceed 100 and should not be read"
              f" as a ceiling.")
    print(f"{enc:22s}{r[1]:9.3f}{rn[1]:9.3f}{pct:10.1%}"
          + (f"{pub:10.1%}" if pub else f"{'(trained)':>11}"))

In [ ]:
# ---------- 4. the held-out encoders ----------
rows = []
for label, (fn, published) in HELD_OUT.items():
    f = DATA_DIR / fn
    if not f.exists():
        print(f"  {label}: cache missing, skipped")
        continue
    NEW = np.load(str(f))["img"].astype(np.float64)[:N_PAIRS]
    W_new = ridge(NEW[tr], HUB_TR)
    P = l2n((NEW[te] @ W_new) @ HEAD)
    nat = ridge(NEW[tr], SPACES["txt_bge"][tr])
    rn = recall(l2n(NEW[te] @ nat) @ GAL.T)
    r = recall(P @ GAL.T)
    pct = r[1] / max(rn[1], 1e-9)
    R = np.random.default_rng(9).standard_normal(W_new.shape) / np.sqrt(NEW.shape[1])
    ctrl = recall(l2n((NEW[te] @ R) @ HEAD) @ GAL.T)[1]
    rows.append((label, published, r[1], rn[1], pct, ctrl))
    print(f"{label:22s}{r[1]:9.3f}{rn[1]:9.3f}{pct:10.1%}"
          + (f"{published:10.1%}" if published else f"{'-':>11}")
          + f"   control {ctrl:.3f}")

In [ ]:
# ---------- 5. THE GATE ----------
sig = next((r for r in rows if r[1]), None)
cnv = next((r for r in rows if not r[1]), None)
print("\n" + "=" * 70)
if sig is None:
    print("SigLIP cache missing - the gate cannot run, and without it")
    print("ConvNeXt's number has nothing to calibrate against.")
else:
    off = sig[4] - sig[1]
    print(f"SigLIP: {sig[4]:.1%} against a published {sig[1]:.1%}  "
          f"(offset {off:+.1%}, tolerance +/-{TOL:.1%})")
    if abs(off) <= TOL:
        print("\nPROTOCOL REPRODUCED. The original construction is recovered,")
        print("so these are LEVELS, not orderings, and they are directly")
        print("comparable to the 93.8-96.5 per cent band in C.13.2.")
        if cnv:
            v = cnv[4]
            # ABOVE and BELOW the band mean opposite things. An earlier
            # version tested only "outside" and sent both to the
            # architecture-costs-something message, which inverts the
            # conclusion when the value is above.
            if v > BAND[1]:
                print(f"\nConvNeXt: {v:.1%} - ABOVE the within-family band "
                      f"({BAND[0]:.1%}-{BAND[1]:.1%}).")
                print("A convolutional encoder transfers BETTER than every")
                print("encoder the band was measured on. Architecture does not")
                print("bound the claim - it does not even cost anything here.")
                print("Do not overclaim: this is one convnet on one corpus, and")
                print("a single point above a band of three is not a trend.")
            elif v >= BAND[0]:
                print(f"\nConvNeXt: {v:.1%} - INSIDE the band. A convolutional")
                print("encoder is a full citizen of the hub; architecture does")
                print("not bound the claim.")
            else:
                print(f"\nConvNeXt: {v:.1%} - BELOW the band. Architecture")
                print("costs something measurable; report the gap rather than")
                print("rounding it in.")
            print(f"control at {cnv[5]:.3f} against chance {1/len(te):.3f}.")
        print("\nAppendix E.1's protocol-offset caveat can be withdrawn, and")
        print("E.2 rewritten as a level. Keep the reconstruction account in")
        print("E.1 as the record of how the number was first obtained.")
    else:
        print("\nNOT REPRODUCED. Some detail still differs from the original.")
        print("Do NOT withdraw the E.1 caveat - the calibrated comparison")
        print("(ConvNeXt vs SigLIP measured identically) remains the honest")
        print("statement, and it already supports the conclusion.")
        print("Check first: does N_PAIRS match the original run, and is the")
        print("txt_bge prefix the same one the DINOv2 caches used?")

In [ ]:
# ---------- 6. persist, so nothing downstream has to re-derive it ----------
# The original run kept the hub in memory and the Colab VM recycled it,
# which is the whole reason this notebook exists. Writing it out is the
# one-line fix for that, and G8 reads the spectrum from here.
out = DATA_DIR / "hub_exact.npz"
np.savez_compressed(
    out,
    sv=_sv, basis=BASIS, mu=_mu, hub_train=HUB_TR,
    scale_per_space=np.array([SPACES[k][tr].std(0).mean() for k in SPACES]),
    space_names=np.array(list(SPACES)),
    space_dims=np.array([SPACES[k].shape[1] for k in SPACES]),
    train_idx=tr, eval_idx=te, n_pairs=N_PAIRS,
    hub_dim=HUB_DIM, alpha=ALPHA, train_on=np.array([TRAIN_ON]),
    head=HEAD,
    **{f"to_hub_{k}": TO_HUB[k] for k in SPACES},
    **{f"space_{k}": SPACES[k] for k in SPACES},
)
print(f"\nwrote {out.name}")
print(f"  sv               {_sv.shape}   <- G8 reads this")
print(f"  basis            {BASIS.shape}")
print(f"  hub_train        {HUB_TR.shape}")
print(f"  spaces           {', '.join(SPACES)}")
print("  Note the spectrum here is of the PER-SPACE SCALED concat, which is")
print("  the object the transfer curve was measured on - not the raw concat")
print("  the seven-space reconstruction produced.")

print("\nScope unchanged by any of this: ConvNeXt is ImageNet-22k")
print("supervised where DINOv2 is self-supervised, so architecture and")
print("objective remain confounded.")